# Example 3: Multi-Source Fusion

Combine AIS data from multiple providers with configurable merge strategies.

When multiple sources cover the same area, Neptune fuses them using
configurable merge strategies. This notebook shows how fusion works
and how to inspect the fusion configuration.

## Prerequisites

```bash
pip install neptune-ais[sql,notebooks]
```

**Data requirement:** This example requires data from at least two sources (NOAA and DMA).
Run `Neptune.download()` for both sources first, or adapt the `sources` list to match
what you have available locally.

## Imports

In [1]:
from neptune_ais.api import Neptune

## Merge Modes

Neptune supports three merge strategies when combining data from multiple sources:

| Mode | Behavior |
|---|---|
| `"best"` | Deduplicate across sources, prefer higher-quality data based on QC scores |
| `"union"` | Keep all records from all sources, tag with provenance |
| `"prefer:<source>"` | Deterministic source preference (e.g., `"prefer:noaa"`) |

## Configure and Download

Pass multiple sources and a merge mode to the `Neptune` constructor.
The date range can be a single date or a `(start, end)` tuple.

Calling `.download()` fetches data from each source. If you already have
local data from [Example 02](02_archival_ingest.ipynb), this will reuse it.

In [2]:
n = Neptune(
    ("2024-06-15"),
    sources=['dma', 'noaa'],
    merge="best",
    cache_dir="/tmp/neptune_demo",
)
n.download()

['positions/dma/2024-06-15', 'positions/noaa/2024-06-15']

## Inspect Fusion Configuration

`.fusion_info()` returns a dictionary describing the active fusion mode,
source precedence order, and per-source statistics.

In [3]:
info = n.fusion_info()
print(f"Mode:         {info['fusion']['mode']}")
print(f"Sources:      {info['sources']}")
print(f"Precedence:   {info['fusion']['source_precedence']}")
print(f"Multi-source: {info['multi_source']}")

Mode:         best
Sources:      ['dma', 'noaa']
Precedence:   ['dma', 'noaa']
Multi-source: True


## Per-Source Breakdown

See how many rows and partitions each source contributes before fusion.

In [4]:
for sd in info["per_source"]:
    print(f"  {sd['source']}: {sd['rows']:,} rows across {sd['partitions']} partitions")

  dma: 19,279,718 rows across 1 partitions
  noaa: 9,985,832 rows across 1 partitions


## Query Fused Data

When you call `.positions()` on a multi-source Neptune instance,
deduplication happens automatically according to the configured merge mode.

In [5]:
positions = n.positions()
df = positions.collect()
print(f"Total fused positions: {len(df):,}")
print(f"Sources in result: {df['source'].unique().to_list()}")

Total fused positions: 14,590,342
Sources in result: ['noaa', 'dma']


## Next Steps

- **[04 — Event Detection](04_event_detection.ipynb)**: Derive port calls, encounters, and more from fused positions
- **[05 — Streaming Pipeline](05_streaming_pipeline.ipynb)**: Connect to live AIS feeds